# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!pip -q install datasets duckdb pandas pyarrow huggingface_hub

In [15]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [16]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [17]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [18]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [19]:
con.sql("""
SELECT COUNT(*)
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

Unit of analysis : One row represents the daily performance of one content page (content_hash_id) for one client (client_hash_id) on one report_date.

Time window : I use March 2026 (month = '2026-03') as the mid-panel analysis month. I use this month rather than the final June 2026 month so that the final month remains a sealed test period.

In [20]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [21]:
con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

## 2. Fields: feature / label / context / excluded

### Features

* `gsc_impressions` — historical search demand for the content.
* `gsc_clicks` — historical organic traffic from Search.
* `gsc_avg_position` — historical average search ranking.
* `ga4_pageviews` — historical page traffic.
* `ga4_engaged_sessions` — historical engaged traffic.

### Label

* **Future content opportunity / refresh outcome** — derived from the subsequent period. It is not available at the decision moment and is therefore kept separate from the input features.

### Context

* `client_hash_id` — identifies the client/site.
* `content_hash_id` — identifies the content page.
* `report_date` — identifies the daily observation date.
* `month` — used to select the analysis window.
* `client_has_gsc` — indicates whether the client has Search Console data.
* `client_has_ga4` — indicates whether the client has Analytics data.
* `gsc_data_available` — indicates whether Search Console data is available for the observation.
* `ga4_data_available` — indicates whether Analytics data is available for the observation.

### Excluded

* `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, and `ai_other` — excluded because these are narrower AI-referral breakdowns that are not necessary for the initial refresh-opportunity feature set.

* `sessions_direct`, `sessions_referral`, `sessions_social`, and `sessions_paid` — excluded because they represent non-organic traffic channels and are not directly relevant to identifying organic content refresh opportunities.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [23]:
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

In [24]:
con.sql("""
SELECT
    COUNT(*) AS rows_with_gsc_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┐
│ rows_with_gsc_available │
│          int64          │
├─────────────────────────┤
│                 3611061 │
└─────────────────────────┘

## 4. Data limits

This data cannot provide a complete and equally reliable history for every client and content page. GSC availability varies across observations, so some rows have Search Console data while others do not. The early history can therefore be unbalanced across clients and pages. In addition, daily observations may overlap with the historical windows used to construct future labels, so the feature window and outcome window must be separated carefully to avoid leakage.


## Self-check

* [x] Every section above is filled — markdown reasoning is included and claims are supported by the relevant query outputs.
* [ ] The notebook runs top to bottom with no errors (`Runtime → Run all`).
* [x] No client names, URLs, or private queries are included.
* [x] Claims use careful words such as observed, measured, directional, and decision-support.
* [ ] The notebook is committed to the repository under `work/notebooks/`.
* [ ] The repository URL will be submitted on the internship card after the final notebook review.
